![Imgur](https://i.imgur.com/A0T6cjl.png)
### [<font size = 2 color="#5890FF">SERIES 4</font>](https://www.kaggle.com/samybaladram/golf-data-analysis-table-of-contents)<font size = 2 color="#7B8698">　‣　C L A S S I F I C A T I O N　🌦️<br></font>[Dummy Classifier](https://www.kaggle.com/samybaladram/04-1-classification-dummy-classifier)・[kNN Classifier](https://www.kaggle.com/samybaladram/04-2-classification-knn-classifier)・[Naive Bayes](https://www.kaggle.com/code/samybaladram/04-3-classification-naive-bayes-classifier)・**Decision Tree**・[Logistic Regression](https://www.kaggle.com/samybaladram/04-5-classification-logistic-regression)<p>
---

# About **Decision Tree**
![Imgur](https://i.imgur.com/hzTrvxr.png)

# Main Mechanism
![Imgur](https://i.imgur.com/EUq2IVu.png)

# 📊　Dataset & Libraries Used
![Imgur](https://i.imgur.com/0kF8nSS.png)

In [ ]:
# IMPORTING DATASET #
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
plt.rcParams['figure.dpi'] = 120

golf_dataset = pd.read_csv("Data/golf_dataset_mini_original_with_testset.csv")
df = golf_dataset.copy()

df

# Algorithm Steps

# 　Preparation Steps
![Imgur](https://i.imgur.com/QX9SUud.png)

In [ ]:
from sklearn.model_selection import train_test_split

df = golf_dataset.copy()

# Set feature matrix X and target vector y
X, y = df.drop('Play', axis=1), df['Play']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, shuffle=False)

![Imgur](https://i.imgur.com/ADYjGyE.png)

# 　Training Phase
![Imgur](https://i.imgur.com/9OjWWv6.png)
![Imgur](https://i.imgur.com/Q8ebgFr.png)

In [ ]:
def sort_attr_label(attr, lbl):
    return (pd.concat([attr, lbl], axis=1)
            .sort_values([attr.name, lbl.name])
            .reset_index()
            .rename(columns={'index': 'ID'})
            .set_index('ID'))

print(sort_attr_label(X_train['Outlook_sunny'], y_train))
print(sort_attr_label(X_train['Temperature'], y_train))

![Imgur](https://i.imgur.com/vFsjrXt.png)

In [ ]:
def potential_split_points(attr):
    sorted_attr = np.sort(attr)
    unique_values = np.unique(sorted_attr)
    return [(unique_values[i] + unique_values[i+1]) / 2 for i in range(len(unique_values) - 1)]

print(potential_split_points(X_train['Outlook_sunny']))
print(potential_split_points(X_train['Temperature']))

![Imgur](https://i.imgur.com/4cNS52O.png)

In [ ]:
from fractions import Fraction

def gini_impurity(labels):
    counts = np.unique(labels, return_counts=True)[1]
    n = len(labels)
    p = counts / n
    formula = '1 - ' + ' - '.join([f'({Fraction(c).limit_denominator()}/{n})²' for c in counts])
    gini = 1 - np.sum(p ** 2)
    return counts, sum(counts), formula, gini.round(3)

def gini_by_attribute(split_point, attr, labels):
    # Define ranges based on the split point
    low_range = f"≤ {split_point}"
    high_range = f"> {split_point}"

    # Divide data based on the split point
    low_data = labels[attr <= split_point]
    high_data = labels[attr > split_point]

    # Calculate Gini impurities for each subset
    data = [(low_range, *gini_impurity(low_data)),
            (high_range, *gini_impurity(high_data))]

    return pd.DataFrame(data, columns=['Value', 'Σ0, Σ1', 'Σ', 'Gini_Formula', 'Impurity'])

print(gini_by_attribute(0.5, X_train['Outlook_sunny'], y_train))

![Imgur](https://i.imgur.com/ewLLEZC.png)

In [ ]:
def potential_split_points(attr):
    sorted_attr = np.sort(attr)
    unique_values = np.unique(sorted_attr)
    return [(unique_values[i] + unique_values[i+1]) / 2 for i in range(len(unique_values) - 1)]

split_points = potential_split_points(X_train['Temperature'])

for point in split_points:
  print(f"Split Point: {point}")
  print(gini_by_attribute(point, X_train['Temperature'], y_train))
  print("\n")

![Imgur](https://i.imgur.com/nHc8qlq.png)

In [ ]:
def weighted_avg_gini(split_point, attr, labels):
    gini_df = gini_by_attribute(split_point, attr, labels)
    total_count = len(labels)
    weighted_ginis = [row['Σ0, Σ1'] * row['Impurity'] for _, row in gini_df.iterrows()]
    weighted_gini = sum(weighted_ginis) / total_count
    formula = ' + '.join([f'({row["Σ"]}/{total_count})*{row["Impurity"]}' for _, row in gini_df.iterrows()])

    output_df = pd.DataFrame({
        'Σ': [gini_df['Σ'].tolist()],
        'Impurity': [gini_df['Impurity'].tolist()],
        'Weighted_Gini_Formula': [formula],
        'Weighted_Gini': [sum(weighted_gini).round(3)]
    })

    return output_df

weighted_avg_gini(0.5, X_train['Outlook_sunny'], y_train)

![Imgur](https://i.imgur.com/PKlvXUP.png)

In [ ]:
def evaluate_splits(df, labels):
    result = []
    for attr in df.columns:
        split_points = potential_split_points(df[attr])
        for point in split_points:
            weighted_gini_df = weighted_avg_gini(point, df[attr], labels)
            result.append({
                'Attribute': attr,
                'Split_Point': point,
                'Weighted_Gini': weighted_gini_df['Weighted_Gini'].values[0]
            })
    return pd.DataFrame(result)

results_df = evaluate_splits(X_train, y_train)
results_df

![Imgur](https://i.imgur.com/k5uzSeo.png)

In [ ]:
sort_attr_label(X_train['Outlook_overcast'], y_train)

![Imgur](https://i.imgur.com/cw1Tw7C.png)

In [ ]:
# Second iteration of training
X_train2 = X_train.drop([2, 6, 11, 12])
y_train2 = y_train.drop([2, 6, 11, 12])
results_df2 = evaluate_splits(X_train2, y_train2)
print(results_df2)
print(sort_attr_label(X_train2['Temperature'], y_train2))

![Imgur](https://i.imgur.com/Zkcq0xD.png)
![Imgur](https://i.imgur.com/5w7vJa8.png)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# The whole Training Phase above is done inside sklearn like this
dt_clf = DecisionTreeClassifier()
dt_clf.fit(X_train, y_train)

from sklearn.tree import plot_tree

# Plot the decision tree
plt.figure(figsize=(20, 10))
plot_tree(dt_clf, filled=True, feature_names=X.columns, class_names=['Not Play', 'Play'])
plt.show()

# 　Classification Phase
![Imgur](https://i.imgur.com/9O6U2H7.png)
![Imgur](https://i.imgur.com/v1gDLk9.png)

In [ ]:
# Make predictions
y_pred = dt_clf.predict(X_test)
print(y_pred)

# 　Evaluation Phase
![Imgur](https://i.imgur.com/vEMEk1F.png)

In [ ]:
# Evaluate the classifier
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

# Key Parameters

# 　Depth of the tree
![Imgur](https://i.imgur.com/9F4Fly4.png)

In [ ]:
labels, predictions, accuracies = list(y_test), [], []

k_list = [2, 3, 4, 5]
for k in k_list:
    dt_clf = DecisionTreeClassifier(max_depth=k)
    dt_clf.fit(X_train, y_train)
    y_pred = dt_clf.predict(X_test)
    predictions.append(list(y_pred))
    accuracies.append(accuracy_score(y_test, y_pred).round(4)*100)

df_predictions = pd.DataFrame({'Label': labels})
for k, pred in zip(k_list, predictions):
    df_predictions[f'Depth_{k}'] = pred

df_accuracies = pd.DataFrame({'Accuracy ': accuracies}, index=[f'Depth_{k}' for k in k_list]).T

print(df_predictions)
print(df_accuracies)

# 　Minimum samples split
![Imgur](https://i.imgur.com/goQEySD.png)

In [ ]:
# Initialize lists
labels, predictions, accuracies = list(y_test), [], []

# Loop through different min samples split
k_list = [9, 8, 5, 2]
for k in k_list:
    dt_clf = DecisionTreeClassifier(min_samples_split=k)
    dt_clf.fit(X_train, y_train)
    y_pred = dt_clf.predict(X_test)
    predictions.append(list(y_pred))
    accuracies.append(accuracy_score(y_test, y_pred).round(4)*100)

# Create DataFrames
df_predictions = pd.DataFrame({'Label': labels})
for k, pred in zip(k_list, predictions):
    df_predictions[f'MinSplit_{k}'] = pred

df_accuracies = pd.DataFrame({'Accuracy ': accuracies}, index=[f'MinSplit_{k}' for k in k_list]).T

print(df_predictions)
print(df_accuracies)

# 　Criterion
![Imgur](https://i.imgur.com/L83Yqbz.png)

In [ ]:
# Initialize lists
labels, predictions, accuracies = list(y_test), [], []

# Loop through different criterion
criterions = ['gini','entropy']
for measure in criterions:
    dt_clf = DecisionTreeClassifier(criterion=measure)
    dt_clf.fit(X_train, y_train)
    y_pred = dt_clf.predict(X_test)
    predictions.append(list(y_pred))
    accuracies.append(accuracy_score(y_test, y_pred).round(4)*100)

# Create DataFrames
df_predictions = pd.DataFrame({'Label': labels})
for k, pred in zip(criterions, predictions):
    df_predictions[f'Measure_{k}'] = pred

df_accuracies = pd.DataFrame({'Accuracy ': accuracies}, index=[f'Measure_{k}' for k in criterions]).T

print(df_predictions)
print(df_accuracies)

# Pros and Cons
![Imgur](https://i.imgur.com/VL8FiUo.png)

# 🌟 Decision Tree Classifier Simplified

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X, y = df.drop('Play', axis=1), df['Play']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, shuffle=False)

dt_clf = DecisionTreeClassifier()
dt_clf.fit(X_train, y_train)

y_pred = dt_clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

In [ ]:
from sklearn.tree import plot_tree

# Plot the decision tree
plt.figure(figsize=(20, 10))
plot_tree(dt_clf, filled=True, feature_names=X.columns, class_names=['Not Play', 'Play'])
plt.show()

![Imgur](https://i.imgur.com/M0mwv0l.png)
### [<font size = 2 color="#5890FF">SERIES 4</font>](https://www.kaggle.com/samybaladram/golf-data-analysis-table-of-contents)<font size = 2 color="#7B8698">　‣　C L A S S I F I C A T I O N　🌦️<br></font>[Dummy Classifier](https://www.kaggle.com/samybaladram/04-1-classification-dummy-classifier)・[kNN Classifier](https://www.kaggle.com/samybaladram/04-2-classification-knn-classifier)・[Naive Bayes](https://www.kaggle.com/code/samybaladram/04-3-classification-naive-bayes-classifier)・**Decision Tree**・[Logistic Regression](https://www.kaggle.com/samybaladram/04-5-classification-logistic-regression)<p>

---
    
*Explore [diverse notebooks full of captivating visuals](https://www.kaggle.com/samybaladram/golf-data-analysis-table-of-contents), each tackling unique facets of the field—all based on some mini, yet enriching, golf play datasets. Your hole-in-one guide to mastering Data Science basics.*


### For a complete list of topics and series, see [⛳️ 📋 Data Science with Golf Dataset・Table of Contents](https://www.kaggle.com/samybaladram/golf-data-analysis-table-of-contents).
---

**Your Feedback Matters! 🍃** If you find this notebook series useful, an ⬆️ *upvote* would be appreciated to help others in the community discover it too!    
    